# Stage 4 -- Compiler-era and temporal shift (E3, E4 / RQ3)
Re-runs the deep suite plus the three strongest baselines under S2 (train v0.4-v0.6,
test v0.7-v0.8; CompilerVersion excluded from inputs) and S3 (train <=2021, test >=2022).
**Accelerator: GPU.** **Attach:** `scvd-lib`, `dive-smart-contracts`, `scvd-stage1-eda`.

In [ ]:
import os, sys, json, time
import numpy as np, pandas as pd
LIB = '/kaggle/input/scvd-lib'
if not os.path.isdir(LIB):
    LIB = os.environ.get('SCVD_LIB', os.path.abspath('..'))   # local fallback
sys.path.insert(0, LIB)
import scvd
root, log = scvd.setup_stage('stage4_generalization')
SEEDS = (0,) if scvd.quick() else scvd.RNG_SEEDS
log.info('seeds=%s quick=%s', SEEDS, scvd.quick())

In [ ]:
pre, post, lab, tools = scvd.load_dive(log)
fam = scvd.family_columns(pre, post)
for k, v in fam.items():
    log.info('family %-18s %3d features', k, len(v))
for v in scvd.VIEWS:
    log.info('view F_%-4s %3d features', v, len(scvd.view_columns(fam, v)))

In [ ]:
splits, sdf = scvd.load_splits() or (scvd.build_splits(pre, lab, 0), None)
# S2 excludes the split variable itself from the inputs
fam_s2 = {k: [c for c in v if c != 'CompilerVersion'] for k, v in fam.items()}

In [ ]:
rows = []
res_path = f'{root}/results/stage4_generalization_results.csv'
done = set()
if os.path.exists(res_path):                 # resume after a kernel restart
    done = {tuple(r) for r in pd.read_csv(res_path)[['model', 'view', 'split', 'seed']].values}
    log.info('resuming: %d runs already checkpointed', len(done))
for sp_name, fam_use in [('S2', fam_s2), ('S3', fam)]:
    for view in scvd.VIEWS:
        d = scvd.make_data(pre, post, lab, fam_use, view, *splits[sp_name])
        M = scvd.cooccurrence(d['y_tr'])
        for name in ['LGC', 'LAT', 'BR-trunk', 'RF-BR', 'LGBM-BR', 'CC-ens']:
            for seed in SEEDS:
                if (name, view, sp_name, seed) in done:
                    log.info('skip %s F_%s %s seed%d (checkpointed)', name, view, sp_name, seed)
                    continue
                if name in ('RF-BR', 'LGBM-BR', 'CC-ens'):
                    proba = scvd.fit_classical(name, d['X_tr'], d['y_tr'], seed)  # train ONCE
                    th = scvd.tune_thresholds(d['y_va'], proba(d['X_va']))
                    pt = proba(d['X_te'])
                else:
                    cfg = {'lr': 1e-3, 'dm': 128, 'layers': 3, 'tau': 0.2, 'batch': 256,
                           'epochs': 60 if scvd.quick() else 120}
                    mdl = (scvd.LGC(d['fams'], M, dm=128, tau=0.2) if name == 'LGC'
                           else scvd.LAT(d['fams'], dm=128, layers=3) if name == 'LAT'
                           else scvd.LAT(d['fams'], dm=128, layers=3, trunk_only=True))
                    pt, _h = scvd.train_torch(mdl, d, cfg, log)
                    pv = scvd.predict(mdl, d, 'va')
                    th = scvd.tune_thresholds(d['y_va'], pv)
                m = scvd.all_metrics(d['y_te'], (pt >= th).astype(int), pt)
                rows.append({'model': name, 'view': view, 'split': sp_name, 'seed': seed, **m})
                scvd.append_results(root, 'stage4_generalization', [rows[-1]])  # checkpoint
                log.info('%s %-9s F_%-4s seed%d macro_f1=%.4f', sp_name, name, view, seed, m['macro_f1'])

In [ ]:
summ, sp = scvd.write_summary(root, 'stage4_generalization')
s1 = scvd._find_input({'s3': 'stage3_summary.csv'})
if s1:
    summ = pd.concat([pd.read_csv(s1['s3']), summ], ignore_index=True)
scvd.plot_shift_degradation(summ, root, log)
scvd.tex_table(summ[summ['view'] == 'all'], f'{root}/tables/stage4_shift.tex',
               ['macro_f1', 'micro_f1'], 'Macro-F1 under S1/S2/S3 on F_all.', 'tab:stage4shift')
summ.round(4)

In [ ]:
url = scvd.push_dataset(root, 'stage4_generalization', 'scvd-stage4-generalization', 'SCVD Stage 4: compiler-era and temporal shift experiments', log)
if url:
    print('DATASET:', url)
else:
    print('push skipped (no credentials); artifacts remain under', root)